In [38]:
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [39]:
print("=" * 60)
print("STEP 1 — LOAD & INITIAL INSPECTION")
print("=" * 60)
df_raw = pd.read_csv('pertamina_oil_gas_raw.csv')

STEP 1 — LOAD & INITIAL INSPECTION


In [40]:
print(f"Shape (raw)        : {df_raw.shape}")
print(f"Jumlah kolom       : {df_raw.shape[1]}")
print(f"\nTipe Data:\n{df_raw.dtypes}")
print(f"\nMissing Values:\n{df_raw.isnull().sum()}")
print(f"\nDuplicate Rows     : {df_raw.duplicated().sum()}")
print(f"\nContoh data (5 baris pertama):\n{df_raw.head().to_string()}")

Shape (raw)        : (512, 14)
Jumlah kolom       : 14

Tipe Data:
record_id                        str
date                             str
field_name                       str
region                           str
product_type                     str
operational_status               str
production_volume_bopd       float64
oil_price_usd_bbl            float64
revenue_usd_k                float64
opex_usd_k                   float64
production_efficiency_pct    float64
active_well_count              int64
downtime_hours               float64
co2_intensity_tco2_boe       float64
dtype: object

Missing Values:
record_id                     0
date                          0
field_name                    0
region                        0
product_type                  0
operational_status            0
production_volume_bopd        0
oil_price_usd_bbl             0
revenue_usd_k                20
opex_usd_k                    0
production_efficiency_pct    16
active_well_count             0


In [41]:
df = df_raw.drop_duplicates()
removed = len(df_raw) - len(df)
print(f"Duplikat ditemukan : {removed} baris")
print(f"Baris setelah clean: {len(df)}")

Duplikat ditemukan : 12 baris
Baris setelah clean: 500


In [42]:
print("Contoh format tanggal yang tidak konsisten:")
print(df['date'].sample(8, random_state=1).tolist())

Contoh format tanggal yang tidak konsisten:
['2021-06-27', '2021-05-19', '2021-06-25', '2020-06-12', '2021-12-17', '2021-01-30', '2023-09-17', '2020-01-14']


In [43]:
def parse_flexible_date(date_str):
    """Handle multiple date format inconsistencies"""
    for fmt in ['%Y-%m-%d', '%d/%m/%Y', '%Y/%m/%d']:
        try:
            return datetime.strptime(str(date_str), fmt)
        except ValueError:
            continue
    return pd.NaT
df['date'] = df['date'].apply(parse_flexible_date)
df['date'] = pd.to_datetime(df['date'])

print(f"Invalid dates setelah parsing: {df['date'].isnull().sum()}")
print(f"Date range: {df['date'].min().date()} → {df['date'].max().date()}")

Invalid dates setelah parsing: 0
Date range: 2020-01-02 → 2023-12-31


In [44]:
# Ekstrak fitur waktu
df['year']    = df['date'].dt.year
df['month']   = df['date'].dt.month
df['quarter'] = df['date'].dt.to_period('Q').astype(str)

In [45]:
neg = (df['production_volume_bopd'] < 0).sum()
print(f"Nilai negatif ditemukan: {neg} baris")
print(f"Nilai sebelum fix: {df[df['production_volume_bopd'] < 0]['production_volume_bopd'].tolist()}")

Nilai negatif ditemukan: 5 baris
Nilai sebelum fix: [-6562.5, -10476.2, -7609.6, -5844.7, -7253.1]


In [46]:
df['production_volume_bopd'] = df['production_volume_bopd'].abs()
print(f"Setelah fix: semua nilai positif ✓")

Setelah fix: semua nilai positif ✓


In [47]:
rev_null = df['revenue_usd_k'].isnull().sum()
df['revenue_usd_k'] = df['revenue_usd_k'].fillna(
    (df['production_volume_bopd'] * df['oil_price_usd_bbl'] * 0.001).round(2)
)
print(f"revenue_usd_k ({rev_null} null)")
print(f"  → Imputed: volume × oil_price × 0.001")
print(f"  → Alasan : revenue = fungsi langsung dari produksi dan harga")


revenue_usd_k (20 null)
  → Imputed: volume × oil_price × 0.001
  → Alasan : revenue = fungsi langsung dari produksi dan harga


In [48]:
eff_null = df['production_efficiency_pct'].isnull().sum()
df['production_efficiency_pct'] = (
    df.groupby('field_name')['production_efficiency_pct']
    .transform(lambda x: x.fillna(x.median()))
)
print(f"\nproduction_efficiency_pct ({eff_null} null)")
print(f"  → Imputed: median per lapangan")
print(f"  → Alasan : efisiensi tiap lapangan punya karakteristik berbeda")


production_efficiency_pct (15 null)
  → Imputed: median per lapangan
  → Alasan : efisiensi tiap lapangan punya karakteristik berbeda


In [49]:
co2_null = df['co2_intensity_tco2_boe'].isnull().sum()
median_co2 = df['co2_intensity_tco2_boe'].median()
df['co2_intensity_tco2_boe'] = df['co2_intensity_tco2_boe'].fillna(median_co2)
print(f"\nco2_intensity_tco2_boe ({co2_null} null)")
print(f"  → Imputed: global median ({median_co2:.4f} tCO2/BOE)")
print(f"  → Alasan : metrik ESG, tidak ada konteks per lapangan yang cukup")

print(f"\nTotal null tersisa: {df.isnull().sum().sum()} ✓")


co2_intensity_tco2_boe (10 null)
  → Imputed: global median (0.0439 tCO2/BOE)
  → Alasan : metrik ESG, tidak ada konteks per lapangan yang cukup

Total null tersisa: 0 ✓


In [50]:
print("\n" + "=" * 60)
print("STEP 6 — OUTLIER DETECTION — OPEX (IQR Method)")
print("=" * 60)

Q1  = df['opex_usd_k'].quantile(0.25)
Q3  = df['opex_usd_k'].quantile(0.75)
IQR = Q3 - Q1
upper_fence = Q3 + 3.0 * IQR

outliers = df[df['opex_usd_k'] > upper_fence]
print(f"Q1          : {Q1:,.1f}")
print(f"Q3          : {Q3:,.1f}")
print(f"IQR         : {IQR:,.1f}")
print(f"Upper fence (Q3 + 3×IQR): {upper_fence:,.1f}")
print(f"Outlier ditemukan: {len(outliers)} baris")

# Winsorize — cap di upper fence, bukan dihapus
df['opex_usd_k_raw']  = df['opex_usd_k']
df['opex_usd_k']      = df['opex_usd_k'].clip(upper=upper_fence)
df['is_opex_outlier'] = (df['opex_usd_k_raw'] > upper_fence).astype(int)

print(f"\nStrategi: Winsorizing (capping di {upper_fence:,.1f})")
print(f"Alasan  : Hapus outlier = kehilangan data. Cap = tetap representatif")
print(f"Outlier di-flag di kolom 'is_opex_outlier' untuk audit trail")


STEP 6 — OUTLIER DETECTION — OPEX (IQR Method)
Q1          : 165.5
Q3          : 319.1
IQR         : 153.5
Upper fence (Q3 + 3×IQR): 779.6
Outlier ditemukan: 12 baris

Strategi: Winsorizing (capping di 779.6)
Alasan  : Hapus outlier = kehilangan data. Cap = tetap representatif
Outlier di-flag di kolom 'is_opex_outlier' untuk audit trail


In [51]:
print("\n" + "=" * 60)
print("STEP 7 — FEATURE ENGINEERING")
print("=" * 60)

# EBITDA (proxy: Revenue - OPEX)
df['ebitda_usd_k'] = (df['revenue_usd_k'] - df['opex_usd_k']).round(2)

# OPEX ratio (cost efficiency indicator)
df['opex_ratio_pct'] = ((df['opex_usd_k'] / df['revenue_usd_k']) * 100).round(2)

# Revenue per sumur aktif
df['revenue_per_well_k'] = (df['revenue_usd_k'] / df['active_well_count']).round(2)

# Produksi per sumur (BOPD/sumur)
df['prod_per_well_bopd'] = (df['production_volume_bopd'] / df['active_well_count']).round(1)

# Availability factor (uptime %)
max_hours = 30 * 24  # 720 jam/bulan
df['availability_pct'] = ((1 - df['downtime_hours'] / max_hours) * 100).clip(0, 100).round(1)

engineered = ['ebitda_usd_k','opex_ratio_pct','revenue_per_well_k','prod_per_well_bopd','availability_pct']
for col in engineered:
    print(f"  ✓ {col:30s} avg: {df[col].mean():.2f}")


STEP 7 — FEATURE ENGINEERING
  ✓ ebitda_usd_k                   avg: 355.22
  ✓ opex_ratio_pct                 avg: 42.44
  ✓ revenue_per_well_k             avg: 21.72
  ✓ prod_per_well_bopd             avg: 286.81
  ✓ availability_pct               avg: 97.43


In [52]:
print("\n" + "=" * 60)
print("STEP 8 — STANDARDIZE CATEGORICAL COLUMNS")
print("=" * 60)

for col in ['field_name','region','product_type','operational_status']:
    df[col] = df[col].str.strip().str.title()
    print(f"  {col}: {sorted(df[col].unique())}")



STEP 8 — STANDARDIZE CATEGORICAL COLUMNS
  field_name: ['Cepu', 'Lirik', 'Pendopo', 'Rantau', 'Sangatta', 'Tambun', 'Tanjung']
  region: ['Jawa', 'Kalimantan', 'Sumatra']
  product_type: ['Condensate', 'Crude Oil', 'Lpg', 'Natural Gas']
  operational_status: ['Active', 'Inactive', 'Maintenance']


In [53]:
print("\n" + "=" * 60)
print("STEP 9 — FINAL VALIDATION & EXPORT")
print("=" * 60)

# Validasi
assert df.isnull().sum().sum() == 0,          "❌ Masih ada null!"
assert (df['production_volume_bopd'] >= 0).all(), "❌ Masih ada nilai negatif!"
assert (df['revenue_usd_k'] >= 0).all(),          "❌ Revenue negatif!"

# Pilih kolom final
final_cols = [
    'record_id','date','year','month','quarter',
    'field_name','region','product_type','operational_status',
    'production_volume_bopd','oil_price_usd_bbl',
    'revenue_usd_k','opex_usd_k','ebitda_usd_k',
    'opex_ratio_pct','production_efficiency_pct',
    'active_well_count','revenue_per_well_k','prod_per_well_bopd',
    'downtime_hours','availability_pct',
    'co2_intensity_tco2_boe','is_opex_outlier'
]

df_clean = df[final_cols].sort_values('date').reset_index(drop=True)
df_clean.to_csv('pertamina_oil_gas_clean.csv', index=False)

print(f"\n✅ Clean dataset saved!")
print(f"\n{'='*40}")
print(f"RINGKASAN DATA QUALITY REPORT")
print(f"{'='*40}")
print(f"Raw rows          : {len(df_raw)}")
print(f"Duplikat dihapus  : {removed}")
print(f"Nilai negatif fix : 5")
print(f"Missing imputed   : 46")
print(f"Outlier di-cap    : {df_clean['is_opex_outlier'].sum()}")
print(f"KPI baru dibuat   : 5")
print(f"Final rows        : {len(df_clean)}")
print(f"Final columns     : {len(df_clean.columns)}")
print(f"\nKEY METRICS:")
print(f"  Total Revenue    : ${df_clean['revenue_usd_k'].sum()/1000:,.1f}M")
print(f"  Total EBITDA     : ${df_clean['ebitda_usd_k'].sum()/1000:,.1f}M")
print(f"  Avg Efficiency   : {df_clean['production_efficiency_pct'].mean():.1f}%")
print(f"  Avg OPEX Ratio   : {df_clean['opex_ratio_pct'].mean():.1f}%")
print(f"  Avg Availability : {df_clean['availability_pct'].mean():.1f}%")



STEP 9 — FINAL VALIDATION & EXPORT

✅ Clean dataset saved!

RINGKASAN DATA QUALITY REPORT
Raw rows          : 512
Duplikat dihapus  : 12
Nilai negatif fix : 5
Missing imputed   : 46
Outlier di-cap    : 12
KPI baru dibuat   : 5
Final rows        : 500
Final columns     : 23

KEY METRICS:
  Total Revenue    : $306.2M
  Total EBITDA     : $177.6M
  Avg Efficiency   : 81.2%
  Avg OPEX Ratio   : 42.4%
  Avg Availability : 97.4%


In [54]:
import os
print(os.path.abspath('pertamina_oil_gas_clean.csv'))

/Users/yasminaulia/New Project/pertamina_oil_gas_clean.csv
